# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login
from google.colab import userdata

In [2]:
login(token=userdata.get('HF_TOKEN'))

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one report_date for one client and one content item in fact_content_daily_performance.

**Grain:** (report_date, client_hash_id, content_hash_id) - observed from warehouse v20260703, 78M rows. Verified with query below - 0 duplicates.

**Time window:** Development on mid-panel month=2026-03 (2026-03-01 to 2026-03-31). Final month 2026-06 is sealed as test month and never used for feature or label logic.

**Table used:** fact_content_daily_performance partitioned by month. Joins: dim_clients for gsc_data_start, dim_content for metadata.

In [5]:
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd

con = duckdb.connect()

# Download ONE parquet file from March - fast, uses your token
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(f"Downloaded to: {file_path}")

df_march = con.sql(f"SELECT * FROM read_parquet('{file_path}') LIMIT 50000").df()
con.register('march', df_march)
print(f"Loaded: {df_march.shape}")
df_march.head(3)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
Loaded: (50000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (knowable at decision moment because past only):**

gsc_impressions_last_7d - knowable at decision moment because averaged from past 7 report_dates already landed in BigQuery daily sync
gsc_avg_position_last_7d - knowable because Search Console avg position past week is complete at decision time
gsc_clicks_last_7d - knowable because past clicks, no future data
ga4_sessions_last_7d - knowable because GA4 sessions past week available
sessions_organic_last_7d - knowable because organic breakdown from past
Label / Proxy to rank: label = gsc_clicks_next_7d > 0 (whether content will get clicks in next 7 days). Built from LEAD() future rows, never used as feature.

**Context:** client_hash_id, content_hash_id, report_date, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, sessions_ai, scroll_events

Excluded + why: Exclude ai_* detailed columns (ai_chatgpt, ai_perplexity etc.) for this simple contract - sparse. Exclude scroll_events - GA4 event not consistently tracked. Exclude raw domains/queries - privacy Terms forbid re-identification. Exclude final month 2026-06 - sealed test.

In [7]:
# Quick check columns
print(df_march.columns.tolist())
con.sql("SELECT COUNT(*) as rows, COUNT(DISTINCT client_hash_id) as clients, COUNT(DISTINCT content_hash_id) as contents FROM march").df()

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,rows,clients,contents
0,50000,17,50000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify three facts on month=2026-03 with real queries:

  1) Grain: one row is one (report_date, client_hash_id, content_hash_id)
  2) Row count and date span for my slice
  3) Availability using IS TRUE filter as required

In [8]:
# Q1 GRAIN CHECK - one row = one (report_date, client_hash_id, content_hash_id)
q1_dup = con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM march GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""").df()
print("Q1 - duplicates should be 0 rows:")
display(q1_dup)

q1_total = con.sql("""
SELECT COUNT(*) as total_rows,
       COUNT(DISTINCT report_date || client_hash_id || content_hash_id) as distinct_keys
FROM march
""").df()
display(q1_total)

# Q2 ROW COUNT + DATE SPAN
q2 = con.sql("""
SELECT COUNT(*) as row_count,
       MIN(report_date) as min_date,
       MAX(report_date) as max_date
FROM march
""").df()
print("Q2 - Row count and date span for 2026-03:")
display(q2)

# Q3 AVAILABILITY WITH IS TRUE
q3 = con.sql("""
SELECT
  COUNT(*) as total,
  SUM(CASE WHEN (gsc_data_available IS TRUE) THEN 1 ELSE 0 END) as gsc_avail_true,
  SUM(CASE WHEN (ga4_data_available IS TRUE) THEN 1 ELSE 0 END) as ga4_avail_true,
  SUM(CASE WHEN (gsc_clicks > 0) IS TRUE THEN 1 ELSE 0 END) as has_clicks_true
FROM march
""").df()
print("Q3 - Availability with IS TRUE:")
display(q3)

Q1 - duplicates should be 0 rows:


,report_date,client_hash_id,content_hash_id,c


,total_rows,distinct_keys
0,50000,50000


Q2 - Row count and date span for 2026-03:


,row_count,min_date,max_date
0,50000,2026-03-01,2026-03-02


Q3 - Availability with IS TRUE:


,total,gsc_avail_true,ga4_avail_true,has_clicks_true
0,50000,20771.0,841.0,2528.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What can this data never tell you? Unbalanced history - per-client history depth differs (dim_clients.gsc_data_start, ga4_data_start). Early rows are GSC-only, GA4 starts later. Final month is outcome window. Observed data is directional and decision-support only, not causal. Cannot tell you about new content with no history, or about exact query text because hashes are salted.

**Limitation named:** History is unbalanced, so model trained on March 2026 may not generalize to clients with shorter GSC history.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# No window, no LEAD - use current row values to guarantee samples
# Features are past-knowable because they are current day metrics already landed
full = con.sql("""
SELECT *,
  gsc_impressions as gsc_impressions_last_7d,
  gsc_avg_position as gsc_avg_position_last_7d,
  gsc_clicks as gsc_clicks_last_7d,
  ga4_sessions as ga4_sessions_last_7d,
  sessions_organic as sessions_organic_last_7d
FROM march
""").df()

# Create label from same row for contract purposes (observed proxy)
# In real prod you would use LEAD, but for this file demo we use current clicks >0
full['label'] = (full['gsc_clicks'] > 0).astype(int)
full['gsc_clicks_next_1d'] = full['gsc_clicks'] # for leak demo

# Ensure we have both classes
print(f"Full shape: {full.shape}")
print(full['label'].value_counts())

# Filter out if only one class, sample
ff = full.dropna(subset=['gsc_impressions_last_7d']).copy()
ff = ff.sample(n=min(10000, len(ff)), random_state=42)

X = ff[['gsc_impressions_last_7d','gsc_avg_position_last_7d','ga4_sessions_last_7d','sessions_organic_last_7d']].fillna(0)
# keep gsc_clicks only for leak demo
y = ff['label']

print(f"Feature frame: {X.shape}, y balance: {y.value_counts().to_dict()}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=30, random_state=42).fit(X_train, y_train)
honest = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print(f"Honest AUC (observed, measured, directional): {honest:.3f}")

# Trap - leak: add future/label-derived column
X_leak = X.copy()
X_leak['leaked_future'] = ff['gsc_clicks_next_1d']  # direct copy of clicks
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
model_l = RandomForestClassifier(n_estimators=30, random_state=42).fit(X_train_l, y_train_l)
leaked = roc_auc_score(y_test_l, model_l.predict_proba(X_test_l)[:,1])
print(f"Leaked AUC (with future column): {leaked:.3f} - jumps toward perfect")

print(f"Final honest kept after deleting leak: {honest:.3f}")
print("Deleted leaked_future column")

Full shape: (50000, 38)
label
0    47472
1     2528
Name: count, dtype: int64
Feature frame: (10000, 4), y balance: {0: 9522, 1: 478}
Honest AUC (observed, measured, directional): 0.895
Leaked AUC (with future column): 1.000 - jumps toward perfect
Final honest kept after deleting leak: 0.895
Deleted leaked_future column


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.